[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/solutions/Lab1_Basic_RAG_LlamaParse_Solutions.ipynb)


# Lab 1 SOLUTIONS: Basic RAG with LlamaParse
## CCI Session 6

**Duration:** 15 minutes

### Clinical scenario
> KHCC pediatric oncologists need quick answers from long Wilms tumor guidelines. You parse the PDF, chunk and embed the text, store vectors, and answer with a grounded RAG chain.

### Objectives
- Parse a complex PDF with **LlamaParse** (layout-aware markdown)
- **Chunk** and **embed** for retrieval
- Store in **Chroma** and run a **retrieval QA** chain
- Inspect retrieved chunks (transparency / “citations”)


---
## Setup

Install packages, add **`OPENAI_API_KEY`** and **`LLAMA_CLOUD_API_KEY`** to Colab **Secrets**, then upload **`WT.pdf`**. If imports fail after install, use **Runtime → Restart session**.


In [ ]:
!pip install -q llama-parse llama-index langchain langchain-text-splitters langchain-openai langchain-community langchain-chroma chromadb pypdf

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')

In [ ]:
from google.colab import files
uploaded = files.upload()
PDF_PATH = '/content/WT.pdf'

---
## Cell 1 — Parse with LlamaParse

**LlamaParse** (LlamaCloud) converts the PDF to **markdown** with layout awareness — tables, lists, and headings stay usable for chunking and retrieval.

**Alternatives:** naive `PyPDFLoader` (Cell 2), **Docling**, **Azure Document Intelligence**, or open layout parsers. Trade-offs: cloud cost + data residency vs. local CPU quality.


In [ ]:
from llama_parse import LlamaParse

parser = LlamaParse(
    api_key=os.environ['LLAMA_CLOUD_API_KEY'],
    result_type='markdown',
    verbose=True,
)
documents = parser.load_data(PDF_PATH)

print(f'Number of documents: {len(documents)}')
print(documents[0].text[:1500])

## Cell 2 — Naive PyPDF Comparison

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
naive_docs = PyPDFLoader(PDF_PATH).load()
print(f'Pages (PyPDF): {len(naive_docs)}')
print('--- PyPDF (jumbled) ---')
print(naive_docs[5].page_content[:1500])
print('\n--- LlamaParse (markdown) ---')
print(documents[0].text[1500:3000])

---
## Cell 3 — Chunking for RAG

**Why chunk?** Embeddings and vector search work on *segments* of text, not whole PDFs. Chunks should be big enough to hold a complete fact (e.g. a dosing table row + context) but small enough to fit several in the LLM context.

**Common strategies (LangChain):**
- **`RecursiveCharacterTextSplitter`** (used here): splits on a priority list (`\n\n`, `\n`, `. `, ` `) so paragraphs and sentences stay intact when possible. Good default for clinical prose and markdown from LlamaParse.
- **`CharacterTextSplitter`**: fixed-size windows; fast but can cut mid-sentence — worse for guidelines.
- **Semantic / heading-based splitters**: split on section titles or embeddings; fewer cuts inside facts, more setup work.

**Tuning `chunk_size` and `chunk_overlap`:**
- Larger chunks (e.g. 1500–2000): more surrounding context per hit; fewer total chunks; risk of diluting relevance.
- Smaller chunks (e.g. 400–600): sharper retrieval; more risk of missing cross-sentence dependencies.
- **Overlap** (often 10–20% of chunk size): reduces the chance that a critical sentence sits on a chunk boundary.

This notebook uses **1000 / 200** as a balanced default for guideline text.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

lc_docs = [Document(page_content=d.text, metadata={'source': 'WT.pdf'}) for d in documents]

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(lc_docs)

print(f'Chunks: {len(chunks)}')
print(chunks[10].page_content[:600])

---
## Cell 4 — Embedding models

Embeddings turn each chunk into a **vector** so “similar meaning” is “similar direction” in space.

**OpenAI options (common in teaching / prod):**
- **`text-embedding-3-small`**: good quality, lower cost and dimension (often 1536). **Default choice** for labs.
- **`text-embedding-3-large`**: stronger retrieval on hard queries; higher cost and vector size.
- **`text-embedding-ada-002`**: older; still usable but v3 models are preferred for new work.

**Alternatives:** Cohere, Voyage, open models (e.g. sentence-transformers) for on-prem or offline; pick one and keep the **same** model for index + query.


In [ ]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

---
## Cell 5 — Vector store (Chroma)

A **vector store** holds chunk vectors and runs similarity search (e.g. cosine) at query time.

**Choices:**
- **Chroma** (here): simple local/embedded store; great for Colab and prototypes; can persist to disk (`persist_directory`).
- **FAISS**: in-memory, very fast; popular for benchmarks; less “database” features out of the box.
- **Managed cloud** (Pinecone, Weaviate Cloud, etc.): scaling, ops, and metadata filters for production.

**Retriever `k`:** number of chunks passed to the LLM. Higher `k` → more evidence (and tokens); lower `k` → tighter focus. Clinical QA often starts with **4–8** and tunes with evaluation (Lab 2).


In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./chroma_wt',
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

## Cell 6 — RAG Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a clinical assistant. Answer the question using ONLY the provided context. If unknown, say so.\n\nContext:\n{context}'),
    ('human', '{input}')
])

doc_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, doc_chain)

for q in [
    'What is the standard treatment for Stage III favorable histology Wilms tumor?',
    'What are the most common side effects of vincristine?'
]:
    res = rag_chain.invoke({'input': q})
    print(f'\nQ: {q}\nA: {res["answer"]}')

## Cell 7 — Citations

In [ ]:
q1 = 'What is the standard treatment for Stage III favorable histology Wilms tumor?'
res = rag_chain.invoke({'input': q1})
for i, doc in enumerate(res['context']):
    print(f'--- Chunk {i+1} ---')
    print(doc.page_content[:300])
    print()

## Stretch — different chunk sizes

In [ ]:
for cs in [500, 2000]:
    sp = RecursiveCharacterTextSplitter(chunk_size=cs, chunk_overlap=int(cs*0.2))
    ch = sp.split_documents(lc_docs)
    print(f'chunk_size={cs} → {len(ch)} chunks')